# 구내식당 식수 인원 예측: upweather1
## 전략: 참여율(Ratio) + 앙상블 + 최적 기상 피처 결합
1. **성공 전략 계승**: `upmodel5`의 참여율(Ratio) 예측 및 3종 모델(XGB, LGBM, CatBoost) 앙상블 유지
2. **기상 피처 최적화**: `weather.csv`에서 유의미한 피처(강수량, 적설, 기온, 습도)를 선별하여 결합
3. **데이터 전처리**: 기상 데이터의 결측치(비/눈 안 옴)를 0으로 정제하여 모델의 학습 효율 극대화

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
weather = pd.read_csv('data/weather.csv')
submission = pd.read_csv('data/sample_submission.csv')

def preprocess_with_weather(df, weather_df):
    df = df.copy()
    df['일자'] = pd.to_datetime(df['일자'])
    df['month'] = df['일자'].dt.month
    df['day'] = df['일자'].dt.day
    df['weekday'] = df['일자'].dt.weekday
    
    # 식사 가능자 수 계산
    df['식사가능자수'] = df['본사정원수'] - (df['본사휴가자수'] + df['본사출장자수'] + df['현본사소속재택근무자수'])
    
    # 날씨 데이터 전처리
    w_df = weather_df.copy()
    if '일시' in w_df.columns:
        w_df = w_df.rename(columns={'일시': '일자'})
    w_df['일자'] = pd.to_datetime(w_df['일자'])
    
    # 추천 피처 선별: 기온, 강수량, 습도, 적설
    # NaN인 강수량과 적설은 0(비/눈 안옴)으로 채움
    w_df['강수량'] = w_df['강수량'].fillna(0)
    w_df['적설'] = w_df['적설'].fillna(0)
    
    # 날짜별 평균 기상 정보 산출 (중복 날짜 방지)
    w_df = w_df.groupby('일자')[['기온', '강수량', '습도', '적설']].mean().reset_index()
    
    # 데이터 병합
    df = pd.merge(df, w_df, on='일자', how='left')
    
    # 병합 후 발생할 수 있는 결측치는 평균값으로 보정
    df['기온'] = df['기온'].fillna(df['기온'].mean())
    df['습도'] = df['습도'].fillna(df['습도'].mean())
    
    return df

train_df = preprocess_with_weather(train, weather)
test_df = preprocess_with_weather(test, weather)

train_df['target_lunch'] = train_df['중식계'] / train_df['식사가능자수']
train_df['target_dinner'] = train_df['석식계'] / train_df['식사가능자수']

In [ ]:
# 2. 모델 및 피처 정의
# 추천 기상 피처 포함
features = ['month', 'day', 'weekday', '식사가능자수', '본사출장자수', '본사시간외근무명령서승인건수', 
            '기온', '강수량', '습도', '적설']

xgb_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 6, 'random_state': 42}
lgb_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 6, 'random_state': 42, 'verbosity': -1}
cat_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'depth': 6, 'random_state': 42, 'silent': True}

def get_ensemble_pred(X_train, y_train, X_test):
    m1 = XGBRegressor(**xgb_params)
    m2 = LGBMRegressor(**lgb_params)
    m3 = CatBoostRegressor(**cat_params)
    
    m1.fit(X_train, y_train)
    m2.fit(X_train, y_train)
    m3.fit(X_train, y_train)
    
    p1 = m1.predict(X_test)
    p2 = m2.predict(X_test)
    p3 = m3.predict(X_test)
    
    return (p1 + p2 + p3) / 3

print("날씨 피처 포함 중식 앙상블 학습 중...")
pred_l_ratio = get_ensemble_pred(train_df[features], train_df['target_lunch'], test_df[features])
print("날씨 피처 포함 석식 앙상블 학습 중...")
pred_d_ratio = get_ensemble_pred(train_df[features], train_df['target_dinner'], test_df[features])

In [ ]:
# 3. 결과 복원 및 저장
submission['중식계'] = pred_l_ratio * test_df['식사가능자수']
submission['석식계'] = pred_d_ratio * test_df['식사가능자수']
submission.to_csv('upweather1_submission.csv', index=False)
print("upweather1: 날씨 피처 접목 앙상블 모델 완료")